In [14]:
import pandas as pd
import numpy as np

from SharedModules import input_dir, output_dir, model_dir
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.preprocessing import TargetEncoder, OneHotEncoder, RobustScaler, KBinsDiscretizer, FunctionTransformer, PolynomialFeatures, OrdinalEncoder, StandardScaler

target = "Heart Disease"

X = pd.read_csv(input_dir + "train.csv", index_col=0).iloc[:, :-1]
y = pd.read_csv(input_dir + "train.csv", index_col=0)[target].map({"Absence": 0, "Presence": 1})

category = [
    "Sex",
    "Chest pain type",
    "FBS over 120",
    "EKG results",
    "Exercise angina",
    "Slope of ST",
    "Number of vessels fluro",
    "Thallium"
]

for col in category:
    X[col] = X[col].astype("category")

In [15]:
LinearView = ColumnTransformer(
    [
        ("numerical", RobustScaler(), make_column_selector(dtype_exclude="category")),
        ("category", TargetEncoder(shuffle=True, smooth=10, random_state=42), make_column_selector(dtype_include="category")),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
).set_output(transform="pandas")

DenseView = ColumnTransformer(
    [
        ("numerical", StandardScaler(), make_column_selector(dtype_exclude="category")),
        ("category", TargetEncoder(shuffle=True, smooth=10, random_state=42), make_column_selector(dtype_include="category")),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
).set_output(transform="pandas")

CategoricalView = Pipeline(
    [
        (
            "bins",
            ColumnTransformer(
                [
                    (
                        "numerical",
                        KBinsDiscretizer(n_bins=4, strategy="quantile", quantile_method="averaged_inverted_cdf", encode="ordinal"),
                        make_column_selector(dtype_exclude="category"),
                    ),
                    (
                        "category",
                        OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1),
                        make_column_selector(dtype_include="category"),
                    ),
                ],
                remainder="drop",
                verbose_feature_names_out=False,
            ).set_output(transform="pandas"),
        ),
        ("cats", FunctionTransformer(lambda df: df.astype(int).astype("category"), feature_names_out="one-to-one")),
    ]
)

PolynomialView = Pipeline(
    [
        ("Linear", LinearView),
        ("poly", PolynomialFeatures(degree=2).set_output(transform="pandas")),
    ]
)

SparseView = ColumnTransformer(
    [
        (
            "num_bins",
            KBinsDiscretizer(n_bins=10, quantile_method="averaged_inverted_cdf", encode="onehot"),
            make_column_selector(dtype_exclude="category"),
        ),
        (
            "cat_ohe",
            OneHotEncoder(handle_unknown="ignore"),
            make_column_selector(dtype_include="category"),
        ),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

In [16]:
X_Linear = LinearView.fit_transform(X, y)
X_Dense = DenseView.fit_transform(X, y)
X_Category = CategoricalView.fit_transform(X, y)
X_Polynomial = PolynomialView.fit_transform(X, y)
X_Sparse = SparseView.fit_transform(X, y)

c:\Users\Blanc\DataScientist\Kaggle\.venv\Lib\site-packages\sklearn\preprocessing\_discretization.py:396: UserWarning: Bins whose width are too small (i.e., <= 1e-8) in feature 4 are removed. Consider decreasing the number of bins.
  warnings.warn(
c:\Users\Blanc\DataScientist\Kaggle\.venv\Lib\site-packages\sklearn\preprocessing\_discretization.py:396: UserWarning: Bins whose width are too small (i.e., <= 1e-8) in feature 1 are removed. Consider decreasing the number of bins.
  warnings.warn(
c:\Users\Blanc\DataScientist\Kaggle\.venv\Lib\site-packages\sklearn\preprocessing\_discretization.py:396: UserWarning: Bins whose width are too small (i.e., <= 1e-8) in feature 4 are removed. Consider decreasing the number of bins.
  warnings.warn(


In [17]:
X_Sparse.shape

(630000, 66)